## 将生僻字JSONL数据转换为 LangChain Document

In [ ]:
import json
from langchain_core.documents import Document

chunks_file_path = r"knowledgeBase\pdfParse\cleaned_data\rare_hanzi_integrated.jsonl"

langch_docs = []
with open(chunks_file_path, "r", encoding="utf-8") as f:
    for line in f:
        obj = json.loads(line)
        
        # 提取需要拼接的字段
        hanzi = obj.get("汉字", "")
        duyin = obj.get("读音", "")
        chaizi = obj.get("拆字", [])
        jieshi = obj.get("解释", "")
        chuxian_shuyu = obj.get("出现在术语", [])
        
        # 将拆字列表转换为字符串
        chaizi_str = "、".join(chaizi) if chaizi else ""
        
        # 将出现在术语列表转换为字符串
        chuxian_shuyu_str = "、".join(chuxian_shuyu) if chuxian_shuyu else ""
        
        # 拼接为连贯句子
        page_content_parts = []
        if hanzi:
            page_content_parts.append(f"'{hanzi}'字")
        if duyin:
            page_content_parts.append(f"其为读音：{duyin}")
        if chaizi_str:
             page_content_parts.append(f"可以拆字表示为：{chaizi_str}")
        if jieshi:
            page_content_parts.append(f"意思是：{jieshi}")
        if chuxian_shuyu_str:
            page_content_parts.append(f"出现在术语：{chuxian_shuyu_str}")
        
        # 用分隔符连接各部分
        page_content = "；".join(page_content_parts)
        
        # 构建 metadata（排除已用于 page_content 的字段）
        metadata = {}
        for key, value in obj.items():
            if key not in ["汉字", "读音", "拆字", "解释","出现在术语"]:
                metadata[key] = value
        
        langch_docs.append(
            Document(
                page_content=page_content,
                metadata=metadata
            )
        )

In [ ]:
import pprint
print("该库包含documents数量：" + str(len(langch_docs)))
pprint.pprint(f"{langch_docs[11].page_content}")
print("格式化展示该条metadata：" + pprint.pformat(langch_docs[11].metadata))
print(langch_docs[11].metadata.get("UNICODE"))

### 使用阿里云模型，向量化、归一化

In [ ]:
import os
import numpy as np
from openai import OpenAI
from langchain_core.embeddings import Embeddings

# 设置阿里云API密钥
API_KEY = os.getenv("DASHSCOPE_API_KEY")
if not API_KEY:
    raise ValueError("请设置环境变量 DASHSCOPE_API_KEY")
BASE_URL = "https://dashscope.aliyuncs.com/compatible-mode/v1"
MODEL_NAME = "text-embedding-v4"

# 初始化OpenAI客户端
client = OpenAI(api_key=API_KEY, base_url=BASE_URL)

class AliyunEmbeddings(Embeddings):
    """阿里云文本嵌入模型，返回L2归一化后的向量（便于余弦相似度计算）"""
    def __init__(self, client, model_name="text-embedding-v4", batch_size=10):
        self.client = client
        self.model_name = model_name
        self.batch_size = batch_size

    def _normalize(self, vec):
        """L2归一化"""
        norm = np.linalg.norm(vec)
        return vec / norm if norm > 0 else vec

    def embed_documents(self, texts):
        all_embeddings = []
        for i in range(0, len(texts), self.batch_size):
            batch = texts[i:i+self.batch_size]
            resp = self.client.embeddings.create(model=self.model_name, input=batch)
            batch_embeddings = [self._normalize(np.array(item.embedding)) for item in resp.data]
            all_embeddings.extend(batch_embeddings)
        return all_embeddings

    def embed_query(self, text):
        resp = self.client.embeddings.create(model=self.model_name, input=text)
        vec = np.array(resp.data[0].embedding)
        return self._normalize(vec).tolist()

# 实例化嵌入模型（构建和检索均使用此实例）
aliyun_emb = AliyunEmbeddings(client, model_name=MODEL_NAME, batch_size=10)

### 使用FAISS，向量本地存储

In [ ]:
from langchain_community.vectorstores import FAISS
from tqdm import tqdm  # 需要先安装：pip install tqdm

# 提取文本、元数据和 ID
texts = [doc.page_content for doc in langch_docs]
metadatas = [doc.metadata for doc in langch_docs]
doc_ids = [doc.metadata.get('UNICODE') for doc in langch_docs]

# 分批生成嵌入，并显示进度
batch_size = aliyun_emb.batch_size
embeddings = []

for i in tqdm(range(0, len(texts), batch_size), desc="生成嵌入进度"):
    batch_texts = texts[i:i+batch_size]
    batch_emb = aliyun_emb.embed_documents(batch_texts)  # 返回归一化的 numpy 数组列表
    embeddings.extend(batch_emb)

# 使用预生成的嵌入构建 FAISS 向量库
# 默认创建的 FAISS 索引类型为 IndexFlatL2。这是一种基于欧几里得距离（L2）的精确搜索（Flat）索引，适用于中小规模数据集的精准相似性检索
vectorstore = FAISS.from_embeddings(
    text_embeddings=list(zip(texts, embeddings)),  # (文本, 嵌入向量) 对
    embedding=aliyun_emb,                           # 仍需传入 embedding 对象用于查询
    metadatas=metadatas,
    ids=doc_ids
)

# 保存到本地
folder_path = "knowledgeBase\\chunks\\hanzi_faiss_index"  # 注意 Windows 路径转义或使用正斜杠
vectorstore.save_local(folder_path)
print(f"--- 向量库已成功保存至目录: {folder_path} ---")

### 本地FAISS向量库加载和查找

In [ ]:

from langchain_community.vectorstores import FAISS

index_folder = r"knowledgeBase\chunks\hanzi_faiss_index"       

local_db = FAISS.load_local(
    folder_path=index_folder, 
    embeddings=aliyun_emb,
    index_name="index",#默认为 "index"。它会去文件夹里找 index.faiss 和 index.pkl 两个文件
    allow_dangerous_deserialization=True
) 

### 根据unicode查找对应的汉字信息，支持单条更新
target_unicode = "U+7B0F"

# 直接从 docstore 中检索
if target_unicode in local_db.docstore._dict:
    doc = local_db.docstore.search(target_unicode)
    print(f"找到汉字: {doc.page_content[:30]}...")
    print(f"元数据: {doc.metadata}")

    # 1. 找到该 Unicode 在 FAISS 内部的数字行号
    # 翻转映射表：从 ID 找 行号
    id_to_index = {v: k for k, v in local_db.index_to_docstore_id.items()}
    row_idx = id_to_index.get(target_unicode)

    if row_idx is not None:
        # 2. 直接从 FAISS 矩阵中提取原始向量
        original_vector = local_db.index.reconstruct(row_idx)
        print(f"数据库中存储的原始向量：{original_vector[:10]}")
        print(f"向量维度：{len(original_vector)}")  
else:
    print("未找到该 Unicode 对应的记录")

### 从已保存的FAISS索引加载数据，构建倒排索引和文档向量数组


In [ ]:
import os
import pickle
import numpy as np
import jieba
from langchain_community.vectorstores import FAISS

# 路径设置
base_path = "knowledgeBase/chunks"
faiss_index_path = os.path.join(base_path, "hanzi_faiss_index")
vectors_path = os.path.join(base_path, "vectors.npy")
doc_ids_path = os.path.join(base_path, "doc_ids.npy")
inverted_index_path = os.path.join(base_path, "inverted_index.pkl")
doc_lengths_path = os.path.join(base_path, "doc_lengths.pkl")
stats_path = os.path.join(base_path, "stats.pkl")

# 1. 加载FAISS向量库（使用之前保存的索引）
print("正在加载FAISS索引...")
vectorstore = FAISS.load_local(faiss_index_path, aliyun_emb, allow_dangerous_deserialization=True)

# 2. 提取所有文档向量（从FAISS索引中重建）
index = vectorstore.index
num_vectors = index.ntotal
print(f"索引中包含 {num_vectors} 个向量。")

# FAISS IndexFlatL2 或 IndexFlatIP 均支持 reconstruct_n
vectors = index.reconstruct_n(0, num_vectors)  #reconstruct_n 方法，从索引中重建所有向量。参数 0 表示从第 0 个向量开始，num_vectors 表示重建的数量。
#返回一个形状为 (num_vectors, dim) 的 NumPy 数组
norms = np.linalg.norm(vectors, axis=1, keepdims=True)#计算每个向量的 L2 范数（欧几里得长度），axis=1 表示对每个向量计算，keepdims=True 保持维度以便广播
vectors = vectors / np.where(norms > 0, norms, 1)  # 归一化，防止除零

# 3. 获取文档ID列表和文档内容
# index_to_docstore_id 是 FAISS 内部映射：index_id -> docstore_id
# docstore 是字典：docstore_id -> Document
index_to_doc_id = vectorstore.index_to_docstore_id  # dict {idx: doc_id}
docstore = vectorstore.docstore  # InMemoryDocstore 对象，可以像字典一样操作

doc_ids = []          # 按索引顺序的文档ID列表
langch_docs = []      # 按索引顺序的 Document 对象列表
for idx in range(num_vectors):
    doc_id = index_to_doc_id[idx]
    doc_ids.append(doc_id)
    doc = docstore.search(doc_id)  # 通过 docstore 获取 Document
    langch_docs.append(doc)

# 4. 保存向量数组和文档ID（供后续检索使用）
np.save(vectors_path, vectors)
np.save(doc_ids_path, np.array(doc_ids))
print(f"向量已保存至 {vectors_path}，文档ID已保存至 {doc_ids_path}")

# 5. 构建倒排索引
print("正在构建倒排索引...")
inverted_index = {}        # 初始化 {词: {doc_id: 词频}}字典
doc_lengths = {}           # 舒适化 {doc_id: 文档词数}字典
N = len(langch_docs)

for i, doc in enumerate(langch_docs):
    doc_id = doc_ids[i]
    # 结巴分词
    words = jieba.lcut(doc.page_content)
    length = len(words)
    doc_lengths[doc_id] = length
    
    # 统计词频
    freq = {}
    for w in words:
        freq[w] = freq.get(w, 0) + 1
    
    # 更新倒排索引
    for w, f in freq.items():
        if w not in inverted_index:
            inverted_index[w] = {}
        inverted_index[w][doc_id] = f

# 计算平均文档长度
avg_len = sum(doc_lengths.values()) / N

# 保存倒排索引、文档长度和统计信息
with open(inverted_index_path, 'wb') as f:
    pickle.dump(inverted_index, f)
with open(doc_lengths_path, 'wb') as f:
    pickle.dump(doc_lengths, f)
with open(stats_path, 'wb') as f:
    pickle.dump({'N': N, 'avg_len': avg_len}, f)

print(f"倒排索引构建完成，文档总数：{N}，平均长度：{avg_len:.2f}")

In [11]:
import jieba

input_file_path = r"knowledgeBase\pdfParse\cleaned_data\termsName_forPrompt.text"
# 加载自定义词典
jieba.load_userdict(input_file_path)

# 现在就可以正常分词了
text = "术语小斗八藻井解释突出于主体房屋前的小屋,清式称抱厦。首卷出处11"
words = jieba.cut(text)
print(" / ".join(words))

术语 / 小斗八藻井 / 解释 / 突出 / 于 / 主体 / 房屋 / 前 / 的 / 小屋 / , / 清式 / 称 / 抱厦 / 。 / 首卷 / 出处 / 11


In [5]:
import jieba


# 现在就可以正常分词了
text = "术语小斗八藻井解释突出于主体房屋前的小屋,清式称抱厦。首卷出处11"
words = jieba.cut(text)
print(" / ".join(words))

术语 / 小斗八 / 藻井 / 解释 / 突出 / 于 / 主体 / 房屋 / 前 / 的 / 小屋 / , / 清式 / 称 / 抱厦 / 。 / 首卷 / 出处 / 11
